- **흐름:** RStudio에서 만든 결핵 코호트 `tb_cohort.csv` 를 읽어서, 그 코호트가 가리키는 흉부 X선(CXR) 인스턴스를 불러와 **이미지로 렌더링**.
- CSV 의 컬럼은 `person_id, image_occurrence_id, local_path`이고, `local_path` 는 CXR DICOM 이 저장된 경로(`gs://...`)다.

In [ ]:
# 필요한 패키지 (없으면 설치)
# 이 노트북은 "아무 패키지도 깔려 있지 않은" JupyterLab 환경을 가정한다.
# - pydicom (+ pylibjpeg 계열): DICOM(.dcm) 영상을 읽는다. MIMIC-CXR 은
#   JPEG/JPEG2000 로 압축돼 있어 libjpeg, openjpeg 플러그인이 필요하다.
# - tqdm: 진행률 표시
import importlib, subprocess, sys

def ensure(pip_name, import_name=None):
    """import 가 안 되면 그 패키지만 조용히 설치한다."""
    try:
        importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])

for pip_name, import_name in [
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("tqdm", "tqdm"),
    ("pydicom", "pydicom"),
    ("pylibjpeg", "pylibjpeg"),
    ("pylibjpeg-libjpeg", "libjpeg"),
    ("pylibjpeg-openjpeg", "openjpeg"),
]:
    ensure(pip_name, import_name)

import subprocess, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pydicom
from tqdm.auto import tqdm

plt.rcParams.update({"figure.dpi": 110, "font.size": 9})

In [ ]:
# 코호트 CSV 찾기 (경로 하드코딩 없이)
#
# 기본 안내: RStudio 가 만든 tb_cohort.csv 를 "이 노트북과 같은 폴더"에 올려 두면 된다.
# 다만 혹시 다른 곳(하위 폴더 / 상위 폴더 / outputs 폴더)에 있어도 자동으로 찾는다.
CSV_NAME = "tb_cohort.csv"

def find_cohort_csv(name, start=None):
    """현재 폴더를 기준으로 위/아래를 훑어 코호트 CSV 경로를 돌려준다."""
    start = Path(start or Path.cwd()).resolve()

    # 1) start 및 상위 폴더들을 가까운 곳부터 확인 (흔한 위치: ., outputs, practice/outputs)
    for root in [start, *start.parents]:
        for rel in (name, Path("outputs") / name, Path("practice") / "outputs" / name):
            cand = root / rel
            if cand.exists():
                return cand

    # 2) 못 찾으면 현재 폴더 아래(하위 폴더 포함)를 재귀 검색
    hits = sorted(start.rglob(name))
    if hits:
        return hits[0]

    raise FileNotFoundError(
        "tb_cohort.csv 를 찾지 못했습니다. "
        "RStudio에서 02_build_tb_cohort.R 를 실행해 만든 CSV 를 "
        "이 노트북과 같은 폴더에 올려 주세요."
    )

COHORT_CSV = find_cohort_csv(CSV_NAME)
print("코호트 CSV:", COHORT_CSV)

# CXR DICOM 을 내려받아 둘 로컬 캐시 폴더 (현재 작업 폴더 아래)
CACHE_DIR = Path("cxr_cache").resolve()
CACHE_DIR.mkdir(parents=True, exist_ok=True)

- **RStudio 가 만든 결핵 코호트 불러오기**

In [ ]:
cohort = pd.read_csv(COHORT_CSV, dtype={"person_id": str, "image_occurrence_id": str})
print(f"결핵 코호트: 환자 {cohort.person_id.nunique()}명 / CXR {len(cohort)}장")

print("tb_cohort.csv (RStudio 산출물)")
cohort.head(8)

- **local_path(GCS) 에서 CXR DICOM 내려받기**
- CXR 원본은 GCS 버킷(`gs://...`)에 있다. 코호트에서 몇 장만 골라 `gsutil` 로 로컬 캐시에 내려받는다. (전체를 받지 않고 시연용으로 앞의 몇 장만 사용한다.)
- GCS 인증이 필요하다: `gcloud auth application-default login` (또는 서비스계정). `gsutil` 이 없으면 `gcloud storage` 로, 그것도 없으면 파이썬 `gcsfs` 로 자동 대체한다.

In [ ]:
N_SHOW = 4  # 시연용으로 몇 장만

def gcs_download(gs_uri: str, dst: Path) -> None:
    """gs:// 객체 하나를 dst 로 내려받는다.
    gsutil 을 우선 쓰고, 없으면 gcloud storage, 그래도 없으면 파이썬 gcsfs 로 대체한다."""
    if shutil.which("gsutil"):
        subprocess.run(["gsutil", "-q", "cp", gs_uri, str(dst)], check=True)
    elif shutil.which("gcloud"):
        subprocess.run(["gcloud", "storage", "cp", gs_uri, str(dst)], check=True)
    else:
        ensure("gcsfs", "gcsfs")  # 아무 것도 없는 환경 대비 (pip 자동 설치)
        import gcsfs
        gcsfs.GCSFileSystem().get(gs_uri, str(dst))

def fetch_dicom(gs_uri: str) -> Path:
    """local_path(GCS URI) 를 로컬 캐시로 내려받고 로컬 경로를 반환."""
    dst = CACHE_DIR / Path(gs_uri).name
    if not dst.exists():
        gcs_download(gs_uri, dst)
    return dst

records = cohort.head(N_SHOW).to_dict("records")
for r in tqdm(records, desc="CXR 다운로드"):
    r["local_file"] = fetch_dicom(r["local_path"])
print("다운로드 완료:", CACHE_DIR)

- **pydicom 으로 로드 후 흉부 X선 이미지로 그리기**

In [ ]:
def read_cxr(dcm_path: Path) -> np.ndarray:
    """DICOM 을 읽어 0~1 정규화된 흑백 영상 배열로 반환."""
    ds = pydicom.dcmread(str(dcm_path))
    arr = ds.pixel_array.astype(np.float32)
    # MONOCHROME1 은 흑백이 반전되어 있으므로 뒤집는다
    if ds.get("PhotometricInterpretation") == "MONOCHROME1":
        arr = arr.max() - arr
    arr = (arr - arr.min()) / (np.ptp(arr) + 1e-9)
    return arr

n = len(records)
fig, axes = plt.subplots(1, n, figsize=(3.2 * n, 4))
axes = np.atleast_1d(axes)
for ax, r in zip(axes, records):
    img = read_cxr(r["local_file"])
    ax.imshow(img, cmap="gray")
    ax.set_title(f"person={r['person_id']}\nimg_occ={r['image_occurrence_id']}", fontsize=9)
    ax.axis("off")
# 그림 제목은 한글 폰트가 없어도 깨지지 않도록 ASCII 로 둔다
fig.suptitle("TB cohort chest X-ray (CXR)", fontsize=12)
fig.tight_layout()
plt.show()

- RStudio: `DatabaseConnector` 로 **CDM DB(PostgreSQL)** 에 접속 (실제 병원 환경과 동일)
- RStudio: `condition_occurrence` 로 **결핵 코호트** 구성 후 `image_occurrence` 의 CXR `local_path` 를 붙여 `tb_cohort.csv` 저장
- Jupyter: 그 CSV 의 `local_path`(GCS) 를 **내려받아 CXR 이미지 렌더링**
- `person_id, image_occurrence_id, local_path` 세 컬럼짜리 CSV 하나가 두 실습 환경을 이어 주는 브릿지 역할. 
- `image_occurrence` 의 `local_path` 는 이미 GCS 경로(`gs://...`)로 저장돼 있어 여기서 `gsutil` 로 바로 내려받을 수 있다.